
# Class 2: Data Cleaning, EDA and ML Preparation

This notebook shows the full coding workflow for data cleaning, exploratory data analysis, feature engineering, and exporting the clean ML-ready dataset.

### Project Objective
The objective of this project is to predict **respiratory-related hospital admissions** using air quality indicators, weather variables, location information, population density, and hospital capacity.

### Target Variable
The target variable is **`hospital_admissions`**.

### Machine Learning Task
This is a **regression problem** because the target variable is numerical and represents the number of hospital admissions.

### Scope of Class 1/2
This notebook focuses on:
1. Data loading and understanding  
2. Data quality checking  
3. Exploratory Data Analysis  
4. Feature engineering  
5. Exporting a clean ML-ready dataset for Class 3  

Model training, hyperparameter tuning, comparison, and final model selection are done in the separate **Class 3 notebook**.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 2. Data Acquisition

The dataset contains city-level air quality indicators, weather variables, hospital capacity, population-density type, and hospital admissions.

This project uses the environmental and healthcare-related variables to predict **`hospital_admissions`**.

**Important note:** This project focuses on prediction, not causal inference. The analysis does not claim that air pollution directly causes specific diseases; it uses the variables as predictors of respiratory-related hospital admission burden.

In [ ]:
# 3. Load dataset
possible_files = [
    "../data/air_quality_health_dataset.csv",
    "data/air_quality_health_dataset.csv",
    "air_quality_health_dataset.csv",
    "/mnt/data/air_quality_health_dataset.csv",
    "/mnt/data/air_quality_health_dataset(2).csv"
]

for file in possible_files:
    try:
        df = pd.read_csv(file)
        print(f"Loaded file: {file}")
        break
    except FileNotFoundError:
        continue
else:
    raise FileNotFoundError("Dataset not found. Please check the data/ folder or upload the dataset.")

df.head()


## 4. Data Definition

| Variable | Description |
|---|---|
| `city` | City where the observation was recorded |
| `date` | Observation date |
| `aqi` | Air Quality Index |
| `pm2_5` | Fine particulate matter concentration |
| `pm10` | Coarse particulate matter concentration |
| `no2` | Nitrogen dioxide concentration |
| `o3` | Ozone concentration |
| `temperature` | Temperature |
| `humidity` | Humidity |
| `population_density` | Population-density category |
| `hospital_capacity` | Healthcare capacity indicator |
| `hospital_admissions` | Target variable: number of hospital admissions |

In [ ]:
# 5. Basic dataset structure
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)
print("\nFirst five rows:")
display(df.head())

In [ ]:
# 6. Data quality check
missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

duplicate_count = df.duplicated().sum()

print("Duplicated rows:", duplicate_count)
display(missing_table)

In [ ]:
# 7. Clean data and create date features
clean = df.copy()

# Convert date column
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")

# Drop duplicated rows if any
clean = clean.drop_duplicates().reset_index(drop=True)

# Create time-based features
clean["year"] = clean["date"].dt.year
clean["month"] = clean["date"].dt.month
clean["day"] = clean["date"].dt.day
clean["dayofweek"] = clean["date"].dt.dayofweek
clean["quarter"] = clean["date"].dt.quarter
clean["is_weekend"] = clean["dayofweek"].isin([5, 6]).astype(int)

print("Cleaned data shape:", clean.shape)
print("Date range:", clean["date"].min(), "to", clean["date"].max())
clean.head()

## 8. Important Data Note

The date range in this dataset may extend far into the future. Therefore, time-related analysis should be interpreted carefully. In this project, the date variables are mainly used as structured features for prediction, not as evidence of real-world long-term historical trends.

In [ ]:
# 9. Target variable analysis
print(clean["hospital_admissions"].describe())

plt.figure(figsize=(7, 4))
plt.hist(clean["hospital_admissions"], bins=30)
plt.title("Distribution of Hospital Admissions")
plt.xlabel("Hospital Admissions")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# 10. Key numerical distributions
numeric_cols = ["aqi", "pm2_5", "pm10", "no2", "o3", "temperature", "humidity", "hospital_capacity", "hospital_admissions"]

clean[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.suptitle("Numerical Variable Distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 11. Relationship between pollution indicators and hospital admissions
pollution_cols = ["aqi", "pm2_5", "pm10", "no2", "o3"]

for col in pollution_cols:
    plt.figure(figsize=(6, 4))
    plt.scatter(clean[col], clean["hospital_admissions"], alpha=0.25)
    plt.title(f"{col} vs Hospital Admissions")
    plt.xlabel(col)
    plt.ylabel("Hospital Admissions")
    plt.show()

In [ ]:
# 12. Correlation analysis
corr_cols = [
    "aqi", "pm2_5", "pm10", "no2", "o3",
    "temperature", "humidity", "hospital_capacity", "hospital_admissions"
]

corr = clean[corr_cols].corr(numeric_only=True)
display(corr["hospital_admissions"].sort_values(ascending=False))

plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# 13. Compact time-based EDA
monthly_admissions = clean.groupby("month")["hospital_admissions"].mean()

display(monthly_admissions)

plt.figure(figsize=(7, 4))
monthly_admissions.plot(kind="bar")
plt.title("Average Hospital Admissions by Month")
plt.xlabel("Month")
plt.ylabel("Average Hospital Admissions")
plt.show()

## 14. Feature Engineering

The following engineered features are created for future modeling:

- `pollution_mean`: average of major pollution indicators  
- `pollution_max`: maximum pollution value among major pollution indicators  
- `pm_ratio`: ratio between PM2.5 and PM10  
- Time-based variables: year, month, day, day of week, quarter, weekend indicator  

These features are designed to summarize environmental exposure and temporal patterns without using the target variable.

pollution_mean and pollution_max were created in addition to the original pollutant variables, not as replacements. pollution_mean captures the overall average pollution burden across multiple pollutants, while pollution_max captures the highest pollutant exposure in each observation. The original pollutant variables such as PM2.5, PM10, NO2, and O3 are still retained for modeling.

In [ ]:
# 15. Create engineered features
clean["pollution_mean"] = clean[["aqi", "pm2_5", "pm10", "no2", "o3"]].mean(axis=1)
clean["pollution_max"] = clean[["aqi", "pm2_5", "pm10", "no2", "o3"]].max(axis=1)

pm_ratio = clean["pm2_5"] / clean["pm10"].replace(0, np.nan)
pm_ratio = pm_ratio.clip(upper=10)
clean["pm_ratio"] = pm_ratio.fillna(pm_ratio.median())

engineered_cols = ["pollution_mean", "pollution_max", "pm_ratio"]
clean[engineered_cols].describe()


## 16. ML Preparation

The final ML-ready dataset keeps only useful predictors and the target variable.  
The raw `date` column is kept in the exported file for transparency, but the Class 3 modeling notebook will exclude it and use structured time features instead.

In [ ]:
# 17. Prepare final ML-ready dataset
ml_ready_cols = [
    "city", "population_density", "date",
    "year", "month", "day", "dayofweek", "quarter", "is_weekend",
    "aqi", "pm2_5", "pm10", "no2", "o3",
    "temperature", "humidity", "hospital_capacity",
    "pollution_mean", "pollution_max", "pm_ratio",
    "hospital_admissions"
]

ml_ready = clean[ml_ready_cols].copy()

print("ML-ready dataset shape:", ml_ready.shape)
display(ml_ready.head())

In [ ]:
# 18. Export clean datasets for Class 3 modeling
from pathlib import Path

data_dir_candidates = [Path("../data"), Path("data"), Path(".")]
output_dir = next((path for path in data_dir_candidates if path.exists() and path.is_dir()), Path("data"))
output_dir.mkdir(exist_ok=True)

clean_path = output_dir / "air_quality_health_clean_full.csv"
ml_ready_path = output_dir / "air_quality_health_ml_ready.csv"

clean.to_csv(clean_path, index=False)
ml_ready.to_csv(ml_ready_path, index=False)

print(f"Saved: {clean_path}")
print(f"Saved: {ml_ready_path}")
print("Clean full shape:", clean.shape)
print("ML-ready shape:", ml_ready.shape)


## Class 1/2 Summary

This notebook defined the prediction target clearly as **`hospital_admissions`**, cleaned the dataset, created engineered features, and exported a machine-learning-ready dataset. The next stage, Class 3, will use the exported file to train, tune, compare, and select the final regression model.